# HDFS launcher

Starts the project's HDFS from PyCharm, no terminal needed. It runs in Docker with 1 NameNode and 2 DataNodes, and every block is stored twice (about 2.3 GB of RAM while it runs).

1. Run the first code cell. It opens Docker Desktop if needed, starts HDFS, fills it with the analysis data the first time, and shows the status.
2. Use it: the Spark notebooks and the dashboard pick HDFS up on their own.
3. When you are done, run the last cell. It stops HDFS and closes Docker Desktop, so the RAM is free again. The data is kept.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from IPython.display import Markdown, display

# self-correcting cwd, same convention as the other notebooks
HERE = next(p for p in (Path.cwd(), Path.cwd() / "infra" / "hdfs") if (p / "docker-compose.yml").exists())
ROOT = HERE.parents[1]

# docker may not be on PATH when the notebook starts; Docker Desktop keeps it here
DOCKER_BIN = Path.home() / "AppData/Local/Programs/DockerDesktop/resources/bin"
ENV = {**os.environ, "PATH": f"{os.environ['PATH']};{DOCKER_BIN}"}
DOCKER = shutil.which("docker", path=ENV["PATH"]) or "docker"
COMPOSE = [DOCKER, "compose", "-f", str(HERE / "docker-compose.yml")]


def powershell(script, *args):
    # Run one of the .ps1 files in this folder and print what it says.
    cmd = ["powershell", "-ExecutionPolicy", "Bypass", "-File", str(HERE / script), *args]
    out = subprocess.run(cmd, capture_output=True, text=True)
    lines = [l for l in out.stdout.splitlines() if l.strip()]
    print("\n".join(lines[-8:]))
    if out.returncode != 0:
        print(out.stderr.strip()[-1500:])
        raise RuntimeError(f"{script} failed")


def namenode(*cmd):
    # Run a command inside the NameNode container and return its output.
    out = subprocess.run([*COMPOSE, "exec", "-T", "namenode", *cmd], capture_output=True, text=True, env=ENV)
    return out.stdout + out.stderr


powershell("start_hdfs.ps1")

# The data survives stop and start. Only the first run (or after `docker compose down`) needs filling.
if "No such file" in namenode("hdfs", "dfs", "-ls", "/lake/processed/features/tabular_features.csv"):
    print("HDFS is empty, copying the analysis data in...")
    powershell("load_lake.ps1")

report = namenode("hdfs", "dfsadmin", "-report")
fsck = namenode("hdfs", "fsck", "/lake")
wanted = ("Live datanodes", "Status:", "Average block replication", "Under-replicated")
print()
print("\n".join(l.strip() for l in (report + fsck).splitlines() if l.startswith(wanted) or l.strip().startswith(wanted)))
print(namenode("hdfs", "dfs", "-du", "-h", "-s", "/lake").strip(), "(size, size with both copies, path)")

display(Markdown("**HDFS is running.** NameNode UI: [http://localhost:9870](http://localhost:9870). When you are done, run the last cell."))

---

### Optional checks

In [ ]:
# Spark reads the fact table from HDFS, writes it back as Parquet and reads it again (runs in its own process).
out = subprocess.run([sys.executable, "-I", str(HERE / "verify_hdfs.py")], capture_output=True, text=True)
print("\n".join(l for l in out.stdout.splitlines() if l.startswith(("read", "wrote", "HDFS"))))
if out.returncode != 0:
    print(out.stderr[-1500:])

In [ ]:
# Backup test: stop one DataNode and read the fact table again. It works because the other DataNode has a copy of every block.
subprocess.run([*COMPOSE, "stop", "datanode"], capture_output=True, env=ENV)
print("DataNode 1 stopped.")
read = (
    "import sys; sys.path.insert(0, r'%s'); import exp_spark as S; s = S.session('backup-test', cores=2); "
    "print('rows read:', s.read.csv(S.lake_path('processed', 'features', 'tabular_features.csv'), header=True).count()); S.stop(s)"
    % (ROOT / "scripts" / "modeling")
)
out = subprocess.run([sys.executable, "-I", "-c", read], capture_output=True, text=True)
print("\n".join(l for l in out.stdout.splitlines() if l.startswith("rows read")) or out.stderr[-1500:])
subprocess.run([*COMPOSE, "start", "datanode"], capture_output=True, env=ENV)
print("DataNode 1 started again.")

---

### Done with HDFS?

Run the cell below. It stops HDFS and closes Docker Desktop (a few seconds). The data stays; the first cell brings everything back.

In [ ]:
powershell("stop_hdfs.ps1")